# Aligning MSI directly onto Visium HD (no separate MSI H&E alignment)

This is a variant of `examples/l12_walkthrough.ipynb` for when you don't need the MSI
section's own H&E alignment as an intermediate step. That notebook does two registrations:

1. align the MSI raster grid onto the MSI section's own H&E image (its Step 1), then
2. register the MSI H&E and Visium H&E images to each other (its Step 3).

This notebook does both in one shot: `sw.align_grid` rasterizes the MSI points and registers
that pseudo-image *directly* against the Visium H&E image. You still place landmarks and run
elastic registration, just once (MSI raster preview <-> Visium H&E) instead of twice.

**Trade-off:** in `l12_walkthrough.ipynb`, one MSI-H&E <-> Visium-H&E registration is computed
once and reused for every MSI modality on the slide (metabolomics, lipidomics, ...). Here, each
modality's own point grid gets its own separate registration directly against Visium, since
there's no shared H&E-to-H&E transform left to reuse. This is worth it when the MSI section's
own H&E image is unavailable, low quality, or you just don't need it for anything else.

In [1]:
import sys
sys.path.insert(0, "msi")  # examples/msi/ contains msi_loader.py (MSI vendor CSV parsing)

import tifffile
from msi_loader import MSIdata

import spatialdata_io
import spatialwarp as sw

/home/croizer/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/croizer/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/croizer/.local/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


A couple of steps below open a window where you click points with your mouse. For that to
work in Jupyter, run this cell first to switch to a real window backend instead of the default
static-image mode (`tk` works out of the box with no extra install; use `qt` instead if you
have PyQt5/PySide installed and prefer it).

In [2]:
%matplotlib tk

## Step 1 -- Load the Visium HD data

Loaded first because its H&E image is what the MSI grid gets aligned onto directly, below.

In [3]:
visium_binned_outputs = "/home/croizer/Documents/04_Collaborations/03_Genial/01_Mario/GENIAL_VisiumHD_LAB5954/Reprocessed/L12/outs/binned_outputs/square_016um"
visium_he_image = "/home/croizer/Documents/04_Collaborations/03_Genial/01_Mario/GENIAL_VisiumHD_LAB5954/20250925_HE_VisiumHD_slide/Slide1_A_L12_Edge.tiff"

# fullres_image_file is required: without it, visium_hd() only loads the downscaled
# hires/lowres preview images, which are NOT in the same pixel space as obsm['spatial'].
# bin_size=16 narrows the tables/shapes to just square_016um (images are unaffected by this).
visium_sdata = spatialdata_io.visium_hd(
    visium_binned_outputs,
    dataset_id="L12",
    bin_size=16,
    fullres_image_file=visium_he_image,
)

# loaded once here as a plain array -- this is the image the MSI grid will be registered
# directly onto in Step 3, and it's the same file visium_sdata's own image was built from,
# so the two stay in the same pixel space.
visium_he_array = tifffile.imread(visium_he_image)
visium_sdata

/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


SpatialData object
├── Images
│     ├── 'L12_full_image': DataTree[cyx] (3, 8192, 8006), (3, 4096, 4003), (3, 2048, 2001), (3, 1024, 1000), (3, 512, 500)
│     ├── 'L12_hires_image': DataArray[cyx] (3, 6000, 5864)
│     └── 'L12_lowres_image': DataArray[cyx] (3, 600, 587)
├── Shapes
│     └── 'L12_square_016um': GeoDataFrame shape: (142954, 1) (2D shapes)
└── Tables
      └── 'square_016um': AnnData (142954, 18085)
with coordinate systems:
    ▸ 'downscaled_hires', with elements:
        L12_hires_image (Images), L12_square_016um (Shapes)
    ▸ 'downscaled_lowres', with elements:
        L12_lowres_image (Images), L12_square_016um (Shapes)
    ▸ 'global', with elements:
        L12_full_image (Images), L12_hires_image (Images), L12_lowres_image (Images), L12_square_016um (Shapes)

## Step 2 -- Load the MSI metabolomics data (points + features only)

No MSI H&E image is loaded here at all -- this workflow never needs it.

In [4]:
metabo_intensity_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/L12-E/20251001-Brosch-Sample02-Total Ion Count.csv"
metabo_annotation_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/Metabolomics list.csv"
metabo_region_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistics files + images metabolomics/L12-E/20251001-Brosch-Sample02_regionspots.csv"

msi_metabo = MSIdata(metabo_intensity_csv, metabo_annotation_csv, metabo_region_csv)
metabo_counts = msi_metabo.get_count_formated()
metabo_points_xy = msi_metabo.coordinates[["x", "y"]].values.astype(float)
metabo_features = metabo_counts.drop(columns=["x", "y"])

# this vendor's MSI raster x-axis runs opposite to the image's -- still needs resolving before
# any alignment, exactly as in l12_walkthrough.ipynb (it's a raw-coordinate quirk, independent
# of which image the grid eventually gets aligned onto).
metabo_points_xy[:, 0] = metabo_points_xy[:, 0].max() - metabo_points_xy[:, 0]

print("metabo_points_xy:", metabo_points_xy.shape, " features:", metabo_features.shape)

metabo_points_xy: (88707, 2)  features: (88707, 77)


## Step 3 -- Align the MSI grid directly onto the Visium H&E image

`sw.build_spatialdata(...)` packages the *raw* MSI points against `visium_he_array` (not an MSI
H&E image) as the reference image. `sw.align_grid(...)` then rasterizes the points and registers
that pseudo-image straight against `visium_he_array`: click a point on the Visium H&E (left),
then its match on the rasterized MSI preview (right), for ~5-10 landmarks (use the feature
browser buttons if the summed intensity isn't the clearest picture), then close the window.

The result is an MSI SpatialData object whose `obsm['spatial']` is already expressed in
**Visium's** pixel space -- there is no separate MSI-H&E-space representation at all.

In [5]:
msi_metabo_raw_sdata = sw.build_spatialdata(
    visium_he_array, metabo_points_xy, values=metabo_features, image_key="he", table_key="msi"
)

msi_metabo_sdata = sw.align_grid(
    msi_metabo_raw_sdata,
    image_key="he",
    table_key="msi",
    output_csv="transformed_coordinates_L12_metabo_direct.csv",
    mesh_size=(8, 8),
    number_of_iterations=100,
)
msi_metabo_sdata

INFO     no axes information specified in the object, setting `dims` to: ('c', 'y', 'x')                           


SpatialData object
├── Images
│     └── 'he': DataArray[cyx] (3, 8192, 8006)
└── Tables
      └── 'msi': AnnData (88707, 77)
with coordinate systems:
    ▸ 'global', with elements:
        he (Images)

**Check it worked:** the plot below overlays the aligned MSI spots directly on the Visium H&E
image. Drag the alpha slider: the colored spots should trace out real tissue features, not
float off to one side.

In [6]:
msi_metabo_table = msi_metabo_sdata.tables["msi"]
sw.plot_overlay(
    image=visium_he_array,
    points_xy=msi_metabo_table.obsm["spatial"],
    values=msi_metabo_table.X.sum(axis=1),
)

(<Figure size 1000.49x1000.49 with 2 Axes>,
 <Axes: title={'center': 'Alignment QC — drag the slider to inspect overlap'}>)

## Step 4 -- Match against Visium spots (no further registration needed)

`msi_metabo_sdata`'s points are already expressed in Visium's own pixel space (Step 3 registered
directly against `visium_he_array`, the same image `visium_sdata` was built from), so there's
nothing left to warp -- pass `already_aligned=True` to have `sw.align()` skip its own
registration step entirely and go straight to nearest-neighbor matching + merging.

In [7]:
adata_metabo = sw.align(
    moving=msi_metabo_sdata,
    fixed=visium_sdata,
    already_aligned=True,
    distance_threshold=20.0,
    fixed_table_key="square_016um",
    moving_obsm_key="metabolite",
)
adata_metabo

/home/croizer/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 110408 × 18085
    obs: 'in_tissue', 'array_row', 'array_col', 'location_id', 'region', 'nearest_index', 'nearest_distance'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs', 'spatialwarp'
    obsm: 'spatial', 'spatial_warped', 'metabolite'

In [14]:
adata_metabo.obsm['metabolite'].columns

Index(['Taurine', 'Imidazole-4-acetate / Thymine', 'Dihydrothymine',
       'Pyrroline-hydroxy-carboxylate / Oxoproline', 'Aspartate', 'Malate*',
       'Adenine', 'Hypoxanthine # Threonate*', 'Ethanolamine phosphate',
       'Glutamine*', 'Glutamate*', 'Quinolinate', 'Phosphoenolpyruvate (PEP)',
       'Urate', 'Glycerol 3-phosphate', '4-Hydroxyphenylpyruvate*',
       'Deoxyribose phosphate', 'Glycerophosphoethanolamine',
       'FA 14:0 (Myristate)', 'Asparaginyl-Proline', 'Methionyl-Serine',
       'Cytidine', 'Glutamyl-Taurine', 'FA 16:1 Palmiteoleate-Sapienate',
       'Threoninyl-Histidine', 'FA 16:0 Palmitate',
       'Glucose phosphate / Fructose phosphate*', 'Asparaginyl-Methionine',
       'Glycerate diphosphate', 'FA 17:1 Heptadecenoate',
       'FA 17:0 Heptadecanoate', 'Glutaminyl-Glutamate', 'Glutamyl-Glutamate',
       'FA 18:3 Linolenate', 'FA 18:2 Linoleate', 'FA 18:1 Oleate',
       'FA 18:0 Stearate', 'Ophthalmate', 'FA 19:3 / Androstandiol',
       'FA 20:5 Eicosap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import scanpy as sc
sc.pp.calculate_qc_metrics(adata_metabo, inplace=True, log1p=True)
# map each channel to an RGB color; metabolite columns live in obsm["metabolite"],
# while total_counts (from sc.pp.calculate_qc_metrics) stays in .obs
channels = {
    "4-Hydroxyphenylpyruvate*": ("red", adata_metabo.obsm["metabolite"]['FA 17:1 Heptadecenoate'].values),
    "total_counts": ("green", adata_metabo.obs["total_counts"].values),
}
color_idx = {"red": 0, "green": 1, "blue": 2}

spatial_coords = adata_metabo.obsm["spatial"]
rgb_image = np.zeros((adata_metabo.n_obs, 3))

for label, (color, values) in channels.items():
    values = values.astype(float)
    vmax = np.nanpercentile(values, 99)
    normalized = np.clip(values / vmax, 0, 1) if vmax > 0 else values
    rgb_image[:, color_idx[color]] = normalized

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(spatial_coords[:, 0], -spatial_coords[:, 1], c=rgb_image, s=2, alpha=0.9)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("IF-like overlay: " + " vs ".join(channels.keys()))

legend_elements = [Patch(facecolor=color, label=label) for label, (color, _) in channels.items()]
ax.legend(handles=legend_elements, loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

## Step 5 -- Add a second MSI modality (lipidomics)

Same recipe as Step 2-4, repeated for the lipidomics export. Unlike `l12_walkthrough.ipynb`,
there's no shared H&E-to-H&E registration to reuse here -- each modality's own grid is
registered directly against `visium_he_array` independently.

In [ ]:
lipid_intensity_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/L12-E/20251002_Brosch_Lipids_Neg2_1-Total Ion Count.csv"
lipid_annotation_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/Final-Planque-Lip-Oct25.csv"
lipid_region_csv = "/home/croizer/Documents/04_Collaborations/03_Genial/02_Maria/Prof Mario Brosch - Results October 2025/Statistic files + images lipids negative/L12-E/20251002_Brosch_Lipids_Neg3_regionspots.csv"

msi_lipid = MSIdata(lipid_intensity_csv, lipid_annotation_csv, lipid_region_csv)
lipid_counts = msi_lipid.get_count_formated()
lipid_points_xy = msi_lipid.coordinates[["x", "y"]].values.astype(float)
lipid_features = lipid_counts.drop(columns=["x", "y"])
lipid_points_xy[:, 0] = lipid_points_xy[:, 0].max() - lipid_points_xy[:, 0]

msi_lipid_raw_sdata = sw.build_spatialdata(
    visium_he_array, lipid_points_xy, values=lipid_features, image_key="he", table_key="msi"
)
msi_lipid_sdata = sw.align_grid(
    msi_lipid_raw_sdata,
    image_key="he",
    table_key="msi",
    output_csv="transformed_coordinates_L12_lipid_direct.csv",
    mesh_size=(8, 8),
    number_of_iterations=100,
)

lipid_matched = sw.align(
    moving=msi_lipid_sdata,
    fixed=visium_sdata,
    already_aligned=True,
    distance_threshold=20.0,
    fixed_table_key="square_016um",
    moving_obsm_key="lipid",
)

# lipid_matched may keep a slightly different set of spots than adata_metabo (each analyte's
# own MSI grid can have different coverage/gaps) -- reindex onto adata_metabo's spots so both
# analytes live in one object, filling any gap in lipid coverage with NaN.
adata_metabo.obsm["lipid"] = lipid_matched.obsm["lipid"].reindex(adata_metabo.obs_names)
adata_metabo.obs["nearest_distance_lipid"] = lipid_matched.obs["nearest_distance"].reindex(adata_metabo.obs_names)
adata_metabo

`adata_metabo` is now a regular Visium `AnnData` object holding gene expression, plus the
matched metabolite and lipid intensities in `.obsm["metabolite"]` / `.obsm["lipid"]` -- same
final shape as `l12_walkthrough.ipynb`, just reached with one registration per modality instead
of one shared registration plus a separate per-modality grid alignment.

The plot below is a final visual sanity check: since `already_aligned=True` skips warping,
`obsm['spatial_warped']` here is just each kept Visium spot's own coordinate -- so this mainly
confirms the distance-threshold filter kept a sensible tissue-shaped region, not scattered
leftovers.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(*adata_metabo.obsm["spatial_warped"].T, s=1, alpha=0.3)
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("Visium points kept after MSI distance filter")
plt.show()